# The pipeline step by step

This notebook walks through the Python API of mastpipe on a **simulated** laboratory
night, one stage per section, looking at the objects each stage produces. It needs no
data and runs in under a minute. The same functions, in the same order, are what
`mastpipe quicklook` runs; here they are called by hand so each intermediate result can be
inspected.

The night uses a small fiber layout made in this notebook — three telescopes, each with a
science and a sky fiber, plus three calibration fibers — because the layout is an input
table and that is the simplest way to see what it does.

In [ ]:
import tempfile
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.io import fits

from mastpipe.config.loader import build_effective
from mastpipe.instruments.registry import load_header_map, load_instrument, load_line_table
from mastpipe.sim.simulator import simulate_night

work = Path(tempfile.mkdtemp(prefix="mastpipe-tutorial-"))
night = work / "night"

instrument = load_instrument("highspec")
# a 12-px pitch: give the trace, line and extraction steps half-widths that fit it
effective = build_effective(
    instrument,
    "Mg",
    "main",
    overrides=[{"trace": {"trace_halfwidth": 5}, "lines": {"halfwidth": 5}, "extract": {"halfwidth": 5.0}}],
)
line_table = load_line_table("highspec", "Mg")
params = effective.params

rows = []
y = 40.0
for k, telescope in enumerate(["mast01", "mast02", "mast03"]):
    rows.append(dict(trace=3 * k, role="arc", telescope=None, y_nominal=y, slope_nominal=0.002, aperture=None))
    rows.append(
        dict(trace=3 * k + 1, role="science", telescope=telescope, y_nominal=y + 12, slope_nominal=0.002, aperture=None)
    )
    rows.append(dict(trace=3 * k + 2, role="sky", telescope=telescope, y_nominal=y + 24, slope_nominal=0.002, aperture=None))
    y += 36
layout = pd.DataFrame(rows)
layout

## A simulated night

`simulate_night` writes raw frames with the headers the HighSpec header map understands,
and a truth table. The full arc lights every trace; the calibration-fiber arc lights the
arc traces only; the science frames carry no arc light. Lines in the calibration arc and the
science spectra drift linearly in time from the full arc, a little more at the top of the
detector than at the bottom. Every fiber has its own small perturbation of the dispersion
law, so the truth holds one pixel-to-wavelength map per fiber.

In [ ]:
truth = simulate_night(
    effective,
    layout,
    line_table,
    night,
    seed=1,
    n_science=2,
    n_cal_arc=1,
    cosmic_rays=10,
    fiber_preset="tutorial9",
)
truth.frames[["file", "exposure_type", "arc_kind", "mid_time", "shift_px"]]

In [ ]:
science_file = night / truth.frames.loc[truth.frames.exposure_type == "science", "file"].iloc[0]
data = fits.getdata(science_file).astype(float)
fig, ax = plt.subplots(figsize=(10, 3.5))
lo, hi = np.percentile(data, [5, 99.5])
ax.imshow(data, origin="lower", cmap="gray", vmin=lo, vmax=hi, aspect="auto")
ax.set(xlabel="column (dispersion)", ylabel="row", title="a raw science frame: 6 lit traces, 3 dark calibration fibers")
plt.show()

## Ingest

Nothing downstream of ingest sees a raw header. `scan_folder` classifies every frame through
the instrument's header map into a `FrameRef` whose `FrameMeta` speaks the pipeline's
vocabulary; `load_frame` applies the detector's crop and orientation.

In [ ]:
from mastpipe.io.headers import HeaderMap
from mastpipe.io.ingest import load_frame, scan_folder

header_map = HeaderMap.from_dict(load_header_map("highspec"))
refs = scan_folder(night, "highspec", header_map)
pd.DataFrame(
    {
        "source": [r.meta.source for r in refs],
        "type": [r.meta.exposure_type for r in refs],
        "exposure_s": [r.meta.exposure_time for r in refs],
        "mid_time": [r.meta.mid_time for r in refs],
        "rule": [r.meta.classification_rule for r in refs],
    }
).sort_values("mid_time").reset_index(drop=True)

In [ ]:
by_type = {}
for ref in refs:
    by_type.setdefault(ref.meta.exposure_type, []).append(load_frame(ref, effective.detector))
{kind: len(frames) for kind, frames in by_type.items()}

## Stage 1 — clean

Masters are median stacks with validity keys; `build_calibration_set` turns them into what a
frame needs (bias, dark rate, background), and `calibrate_frame` applies the notebook's
arithmetic and builds the variance and mask. Cosmic rays are flagged on science frames with
a median filter that runs along the dispersion axis so emission-line cores are spared.

In [ ]:
from mastpipe.steps.calib import build_calibration_set, build_master, calibrate_frame
from mastpipe.steps.cosmics import flag_cosmic_rays

bias = build_master(by_type["bias"], "bias", params.calib)
dark = build_master(by_type["dark"], "dark", params.calib)
background = build_master(by_type["background"], "background", params.calib)
calib = build_calibration_set(bias, dark, background, effective.detector)

# the simulator writes its science frames as IMAGETYP 'light'; ingest keeps that provisional type
science_raw = by_type["light"][0]
science = calibrate_frame(science_raw, calib, subtract_background=True, saturation=effective.detector.saturation)
science = replace(
    science,
    mask=flag_cosmic_rays(science.data, science.variance, science.mask, params.calib.cosmic_sigma, params.calib.cosmic_size),
)
print(f"bias level {bias.data.mean():.1f} ADU, dark rate {np.nanmedian(calib.dark_rate):.3f} ADU/s")
print(f"calibrated frame: {science.data.shape}, {int(science.mask.sum())} masked pixels")

The three masters that `build_calibration_set` turned into the calibration set. The bias is
a flat level with read noise, the dark rate is the dark master minus the bias per second of
exposure, and the background carries the stray light that the shutter-closed frames do not.

In [ ]:
def show_image(ax, image, title, cmap="gray", percentiles=(2, 98)):
    lo, hi = np.nanpercentile(image, percentiles)
    im = ax.imshow(image, origin="lower", cmap=cmap, vmin=lo, vmax=hi, aspect="auto")
    ax.set(title=title, xlabel="column", ylabel="row")
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)


fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
show_image(axes[0], bias.data, f"master bias ({bias.n_frames} frames), ADU")
show_image(axes[1], calib.dark_rate, f"dark rate ({dark.n_frames} frames), ADU/s")
show_image(axes[2], background.data, f"master background ({background.n_frames} frames), ADU")
plt.tight_layout()
plt.show()

Before and after `calibrate_frame`, and the mask it left: saturated pixels plus the cosmic
rays `flag_cosmic_rays` found. The simulator planted single-pixel hits at known positions
(circles), so the mask can be checked against them; a hit that lands in a dark region of the
frame is the easiest to spot, one on an emission-line core the hardest to tell from a line.

In [ ]:
planted_hits = np.array(truth.extras["cosmic_rays"][science.meta.source])  # (row, column) pairs
flagged = np.argwhere(science.mask)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
show_image(axes[0], science_raw.data, "raw science frame (ADU)", percentiles=(5, 99.5))
show_image(axes[1], science.data, "calibrated: bias, dark and background removed (ADU)", percentiles=(5, 99.5))
axes[1].plot(flagged[:, 1], flagged[:, 0], "s", ms=3, mfc="none", mec="tab:red", mew=0.8, label="masked pixels")
axes[1].plot(
    planted_hits[:, 1], planted_hits[:, 0], "o", ms=9, mfc="none", mec="tab:cyan", mew=0.8, label="planted cosmic rays"
)
axes[1].legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

hit_set = {tuple(h) for h in planted_hits.tolist()}
found = sum(tuple(f) in hit_set for f in flagged.tolist())
print(f"{found} of {len(planted_hits)} planted cosmic rays are in the mask; {len(flagged) - found} other pixels masked")

## Stage 2 — traces and extraction

Traces are found on the flat master, matched to the layout table and **re-keyed by the
layout's trace ids** (`layout_indexed`); detections that match no row are dropped. Every
later per-fiber object is keyed by those ids.

In [ ]:
from mastpipe.steps.trace import find_traces, layout_indexed, match_layout, relative_throughput

flats = [calibrate_frame(f, calib, subtract_background=params.calib.flat_subtract_background) for f in by_type["flat"]]
flat = build_master(flats, "flat", params.calib)
detected = find_traces(flat.data, params.trace)
traceset = layout_indexed(match_layout(detected, layout, params.trace.layout_tolerance))
throughput = relative_throughput(flats[0], traceset, params.extract.halfwidth)

print(f"{len(detected)} detected, {len(traceset)} matched, missing {traceset.missing}, extra {traceset.extra}")
pd.DataFrame(
    {
        "fiber": [t.index for t in traceset],
        "role": [t.role for t in traceset],
        "telescope": [t.telescope for t in traceset],
        "y_center": [round(t.y_center, 2) for t in traceset],
        "rms_px": [round(t.rms, 3) for t in traceset],
        "throughput": [round(throughput[t.index], 3) for t in traceset],
    }
)

In [ ]:
profile = np.nanmedian(flat.data[:, 700:900], axis=1)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(profile, color="0.3")
for t in traceset:
    ax.axvline(
        t.y_center, color="tab:red" if t.role == "science" else "tab:blue" if t.role == "sky" else "tab:green", alpha=0.6
    )
ax.set(
    xlabel="row",
    ylabel="flat level (ADU)",
    title="collapsed flat profile with the matched traces (green arc, red science, blue sky)",
)
plt.show()

The same traces on the two-dimensional flat: each trace is a polynomial `y(x)` fitted to the
centroids `find_traces` measured in blocks of columns, and the dashed lines are the extraction
aperture (`extract.halfwidth` rows either side) that every later stage will sum over. The
layout's nominal rows are the ticks on the left; `match_layout` slid them as a comb onto the
detections (`layout_shift`) before pairing. The lower panel shows how far the measured centroids
sit from the fitted polynomial, the `rms_px` of the table above.

In [ ]:
role_color = {"arc": "tab:green", "science": "tab:red", "sky": "tab:blue"}
x = np.arange(flat.data.shape[1])

fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), gridspec_kw={"height_ratios": [3, 1.2]})
show_image(
    axes[0],
    flat.data,
    f"master flat (zoomed to the traces) with the {len(traceset)} matched traces, "
    f"layout shift {traceset.layout_shift:+.2f} px",
    percentiles=(5, 99.5),
)
for t in traceset:
    c = role_color.get(t.role, "0.5")
    axes[0].plot(x, t.y(x), color=c, lw=1.0)
    axes[0].plot(x, t.y(x) + params.extract.halfwidth, color=c, lw=0.6, ls="--")
    axes[0].plot(x, t.y(x) - params.extract.halfwidth, color=c, lw=0.6, ls="--")
    axes[0].text(x[-1] + 8, t.y(x[-1]), str(t.index), color=c, va="center", fontsize=8)
for _, row in layout.iterrows():
    axes[0].plot([-25, -5], [row.y_nominal, row.y_nominal], color=role_color[row.role], lw=2)
y_lo, y_hi = min(t.y_center for t in traceset) - 25, max(t.y_center for t in traceset) + 25
axes[0].set(xlim=(-30, x[-1] + 30), ylim=(y_lo, y_hi))

for t in traceset:
    resid = t.y_samples - t.y(t.x_samples)
    axes[1].plot(t.x_samples[t.kept], resid[t.kept], ".", color=role_color.get(t.role, "0.5"), ms=4)
    axes[1].plot(t.x_samples[~t.kept], resid[~t.kept], "x", color="0.6", ms=4)
axes[1].axhline(0, color="0.3", lw=0.5)
axes[1].set(
    xlabel="column", ylabel="centroid − fit (px)", title="centroid residuals about the fitted traces (grey crosses: clipped)"
)
plt.tight_layout()
plt.show()

`relative_throughput` is the flat's aperture sum of every fiber divided by the median over the fibers;
the sky subtraction will scale the sky fiber by the ratio of the two throughputs of a telescope.
The dark calibration fibers carry only ambient light in the flat, so their throughput means
nothing and they are never scaled by it.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.8))
fibers = [t.index for t in traceset]
ax.bar(fibers, [throughput[f] for f in fibers], color=[role_color.get(t.role, "0.5") for t in traceset])
for k, telescope in enumerate(["mast01", "mast02", "mast03"]):
    ax.text(3 * k + 1.5, 1.22, telescope, ha="center", fontsize=9)
ax.axhline(1, color="0.3", lw=0.5, ls=":")
ax.set(
    xlabel="fiber (layout trace id)",
    ylabel="relative throughput",
    ylim=(0, 1.35),
    xticks=fibers,
    title="relative throughput from the flat (green arc, red science, blue sky)",
)
plt.show()

In [ ]:
from mastpipe.steps.extract import extract_aperture

spectra = extract_aperture(science, traceset, params.extract)
fig, ax = plt.subplots(figsize=(10, 3))
for s in spectra:
    if s.telescope == "mast01":
        ax.plot(s.pixel, s.flux, lw=0.8, label=f"fiber {s.fiber} ({s.role})")
ax.set(xlabel="pixel", ylabel="ADU", title="raw aperture extraction, telescope mast01")
ax.legend()
plt.show()

All nine extractions at once, offset for clarity: the six lit fibers show the simulated science
continuum with its absorption lines and the sky lines on top, the three calibration fibers see
nothing but noise, as they should on a science frame.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
step = 1.2 * np.nanpercentile([s.flux for s in spectra if s.role != "arc"], 99)
for k, s in enumerate(spectra):
    offset = k * step
    ax.plot(s.pixel, s.flux + offset, lw=0.6, color=role_color.get(s.role, "0.5"))
    ax.text(s.pixel[-1] + 10, offset, f"{s.fiber} {s.role} {s.telescope or ''}", va="center", fontsize=8)
ax.set(xlabel="pixel", ylabel="ADU + offset", title="every fiber's raw aperture extraction of the science frame", yticks=[])
ax.set_xlim(0, s.pixel[-1] + 160)
plt.show()

What one aperture sums: a cut across the detector (the median of forty columns, to beat the
noise down), with the aperture of each trace shaded. `aperture_weights` gives the rows and the weight of each (a `round` aperture
takes whole rows; `subpixel` splits the edge rows).

In [ ]:
from mastpipe.steps.extract import aperture_weights

col = 800
cut = np.nanmedian(science.data[:, col - 20 : col + 20], axis=1)
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(cut, color="0.2", lw=0.8, drawstyle="steps-mid")
for t in traceset:
    rows, weights = aperture_weights(t.y(col), params.extract.halfwidth, params.extract.aperture)
    ax.axvspan(rows.min() - 0.5, rows.max() + 0.5, color=role_color.get(t.role, "0.5"), alpha=0.2)
    ax.axvline(t.y(col), color=role_color.get(t.role, "0.5"), lw=0.8)
ax.set(
    xlabel="row",
    ylabel="ADU",
    xlim=(y_lo, y_hi),
    title=f"columns {col - 20}–{col + 20}: the calibrated frame across the traces, "
    f"apertures shaded ({params.extract.aperture}, {len(rows)} rows each)",
)
plt.show()

## Stage 3 — wavelengths

### Lines on the full arc

The full arc is calibrated and extracted like any frame, with a continuum fitted so the
cross-correlation between fibers works on continuum-subtracted flux. `find_lines` then
measures every emission line on the two-dimensional strip around each trace, with flags that
say why a line would or would not be used.

In [ ]:
from mastpipe.steps.extract import continuum_subtract_all
from mastpipe.steps.lines import find_lines, lines_summary

arc_ref = next(
    r
    for r in refs
    if r.meta.exposure_type == "arc" and truth.frames.set_index("file").loc[r.meta.source, "arc_kind"] == "full"
)
full_arc = calibrate_frame(load_frame(arc_ref, effective.detector), calib)
arc_spectra = continuum_subtract_all(extract_aperture(full_arc, traceset, params.extract), params.extract)
fluxes = [s.flux - s.continuum for s in arc_spectra]
tables, strips = find_lines(full_arc.data, full_arc.variance, traceset, params.lines)
lines_summary(tables)

Before any line is measured, the arc's raw extraction gets a continuum fit (`fit_continuum`,
a clipped Chebyshev polynomial that the emission lines are rejected from), and the
continuum-subtracted flux is what the cross-correlation between fibers will use. The simulated
lamp has no continuum, so here the fit hugs zero; a real Th-Ar lamp carries one, and the
hollow-cathode glow varies along the spectrum.

In [ ]:
s0 = arc_spectra[0]
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(s0.pixel, s0.flux, lw=0.6, color="0.4", label="raw extraction")
ax.plot(s0.pixel, s0.continuum, lw=1.2, ls="--", color="tab:orange", label="fitted continuum")
top = np.nanpercentile(s0.flux, 97)
ax.set(
    xlabel="pixel",
    ylabel="ADU",
    ylim=(-0.3 * top, 1.6 * top),
    title=f"full arc, fiber {s0.fiber}: raw extraction and its continuum "
    f"(zoomed to the line bases; peaks reach {np.nanmax(s0.flux):.0f} ADU)",
)
ax.legend()
plt.show()

`find_lines` works on a straightened strip around each trace (upper panel, continuum removed
row by row) and collapses it to a profile (lower panel). Each detected line is marked by the
verdict on it: green lines passed every test and will be used for the wavelength fit
(`use_x`); the others say why not — a blend that the deblending could not resolve, a line
too close to the detector edge, or a saturated peak. The flags are columns of the table, so
any other cut can be made later.

In [ ]:
k = 0  # position in the trace set; fiber tables[k] belongs to traceset.traces[k]
table, strip = tables[k], strips[k]
profile = np.nansum(strip, axis=0)

fig, axes = plt.subplots(2, 1, figsize=(12, 5.5), sharex=True, gridspec_kw={"height_ratios": [1, 2]})
lo, hi = np.nanpercentile(strip, [5, 99.8])
axes[0].imshow(
    strip,
    origin="lower",
    aspect="auto",
    cmap="gray",
    vmin=lo,
    vmax=hi,
    extent=(-0.5, strip.shape[1] - 0.5, -params.lines.halfwidth - 0.5, params.lines.halfwidth + 0.5),
)
axes[0].set(
    ylabel="row − trace",
    title=f"fiber {traceset.traces[k].index}: straightened, continuum-free strip the lines are measured on",
)

axes[1].plot(profile, color="0.3", lw=0.7)
seen = set()
for _, line in table.iterrows():
    if line.use_x:
        color, label = "tab:green", "used (use_x)"
    elif line.near_edge:
        color, label = "tab:orange", "near the edge"
    elif line.saturated:
        color, label = "tab:red", "saturated"
    elif not line.on_trace:
        color, label = "tab:brown", "off the trace"
    else:
        color, label = "tab:purple", "blend / fit failed"
    axes[1].axvline(line.x_center, color=color, lw=1.4, alpha=0.7, label=None if label in seen else label)
    seen.add(label)
noise = np.nanmedian(np.abs(profile - np.nanmedian(profile))) * 1.4826
axes[1].set(xlabel="pixel", ylabel="collapsed strip (ADU)", yscale="log", ylim=(3 * noise, profile.max() * 2))
axes[1].legend(fontsize=8, loc="upper right", ncol=3)
plt.tight_layout()
plt.show()
print(f"{len(table)} lines detected, {int(table.use_x.sum())} usable in x, {int(table.use_y.sum())} usable in y")

### The first reference solution

A reference is made once per grating and layout from two anchor pairs (wavelength, pixel)
that a person reads off the anchor panels. Here the anchors come from the simulator's truth.
`solve_full_arc` seeds every other fiber from the anchored one by cross-correlation and then
solves each independently: identify lines through the current model, fit the grating
equation, clip, repeat until the wavelength map stops moving.

On a real night `mastpipe identify` draws this two-panel figure when it has no anchors yet:
the catalogue atlas convolved to the instrument's resolution above (here the synthetic atlas
of the shipped line table, since the Murphy atlas is not needed for the tutorial), the
measured continuum-subtracted arc of the anchor trace below, both normalised and numbered on
their strongest well-separated lines. A person matches the patterns, picks two pairs, and
types them back as anchors. Here the anchors come from the simulator's truth instead.

In [ ]:
from IPython.display import Image, display

from mastpipe.io.qa import plot_anchor_panels
from mastpipe.steps.atlas import convolve_to_resolving_power, synthetic_atlas

band = effective.band_nm
catalogue = pd.DataFrame({"wavelength": line_table["wavelength_nm"], "logI": line_table["logI"]})
atlas_wave, atlas_flux = synthetic_atlas(catalogue, (band[0] - 1.0, band[1] + 1.0), effective.resolving_power)
atlas_conv, _ = convolve_to_resolving_power(atlas_wave, atlas_flux, effective.resolving_power)
usable = pd.DataFrame(
    {
        "wavelength": line_table["wavelength_nm"],
        "species": line_table["species"],
        "usable": line_table["usable"].to_numpy(dtype=bool),
    }
)
template, measured = plot_anchor_panels(
    atlas_wave, atlas_conv, usable, fluxes[0], tables[0], band, out=work / "anchor_panels.png"
)
display(Image(filename=str(work / "anchor_panels.png"), width=900))
pd.concat(
    [template.rename(columns={"wavelength": "template nm"}), measured.rename(columns={"x_center": "measured px"})], axis=1
)

In [ ]:
from mastpipe.steps.atlas import catalogue_arrays
from mastpipe.steps.geometry import grating_geometry
from mastpipe.steps.wavecal import build_reference, fiber_agreement, solve_full_arc
from mastpipe.steps.wavemodel import GratingModel, anchor_model

n_pix = full_arc.data.shape[1]
geom = grating_geometry(effective.geometry, effective.detector.pixel_size_um)
cat_wl, cat_sp = catalogue_arrays(line_table, params.wavecal.usable_only)

# two bright, well-separated, on-chip catalogue lines of fiber 0, at their true pixels
bright = np.where(truth.lines_on_chip & (truth.line_logI > 2.0))[0]
i1, i2 = bright[0], bright[-1]
pairs = [
    (float(truth.lines_nm[i1]), float(truth.line_pixels[0][i1])),
    (float(truth.lines_nm[i2]), float(truth.line_pixels[0][i2])),
]
print("anchors:", [(round(w, 4), round(x, 2)) for w, x in pairs])

model0 = anchor_model(pairs, geom, n_pix)
solutions = solve_full_arc(tables, fluxes, traceset, model0, cat_wl, cat_sp, params.wavecal, ref_fiber=0, geom=geom)
reference = build_reference(
    "highspec", "Mg", "tutorial9", solutions, created=arc_ref.meta.mid_time, source_checksums=(arc_ref.meta.checksum,)
)
print("reference", reference.identifier, "valid from", reference.valid_from)
pd.DataFrame(
    {
        "fiber": list(solutions),
        "lines": [s.n_lines for s in solutions.values()],
        "rms_px": [round(s.rms_px, 4) for s in solutions.values()],
    }
)

In [ ]:
from mastpipe.steps.wavemodel import model_from_solution

x = np.arange(n_pix)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for fiber, sol in solutions.items():
    model = model_from_solution(sol, geom)
    # every fiber is compared with its own simulated dispersion law
    truth_model = GratingModel(
        x_c=truth.x_c_per_trace[fiber], B=truth.B_per_trace[fiber], alpha=geom.alpha_rad, poly=None, n_pix=n_pix, geom=geom
    )
    axes[0].plot(sol.lines.x, sol.lines.dx_px, ".", label=f"fiber {fiber}")
    axes[1].plot(x, (model.wavelength(x) - truth_model.wavelength(x)) / model.dispersion(x), lw=0.8, label=f"fiber {fiber}")
axes[0].set(xlabel="pixel", ylabel="fit residual (px)", title="identified lines: residuals")
axes[1].set(xlabel="pixel", ylabel="(solved − truth) / dispersion", title="solution against the simulator's truth")
axes[0].legend(fontsize=7, ncol=3)
plt.tight_layout()
plt.show()
fiber_agreement(solutions).head()

The solution of the anchored fiber: every identified line labelled with its catalogue
wavelength on the arc spectrum, and the dispersion curve, wavelength against pixel, with the
two anchors it started from and the identified lines on it.

In [ ]:
sol0 = solutions[0]
model_fit = model_from_solution(sol0, geom)
fig, axes = plt.subplots(2, 1, figsize=(12, 6.5), gridspec_kw={"height_ratios": [1.6, 1]})
axes[0].plot(fluxes[0], color="0.3", lw=0.7)
ymax = np.nanmax(fluxes[0])
for _, line in sol0.lines.iterrows():
    axes[0].axvline(line.x, color="tab:green", lw=0.5, alpha=0.6)
    axes[0].text(
        line.x, ymax * 1.05, f"{line.wavelength:.3f}", rotation=90, fontsize=6.5, ha="center", va="bottom", color="tab:green"
    )
for w, px in pairs:
    axes[0].axvline(px, color="tab:red", lw=1.5, ls=":", label=f"anchor {w:.4f} nm @ {px:.1f} px")
axes[0].set(
    ylabel="continuum-subtracted ADU",
    title=f"fiber 0: {sol0.n_lines} identified lines (green, catalogue nm) and the two anchors",
    ylim=(-0.05 * ymax, ymax * 1.6),
)
axes[0].legend(fontsize=8, loc="upper left")

axes[1].plot(x, model_fit.wavelength(x), color="0.2", lw=1, label="fitted grating model")
axes[1].plot(sol0.lines.x, sol0.lines.wavelength, ".", color="tab:green", ms=5, label="identified lines")
axes[1].plot([p[1] for p in pairs], [p[0] for p in pairs], "o", mfc="none", mec="tab:red", ms=10, mew=1.5, label="anchors")
axes[1].set(
    xlabel="pixel",
    ylabel="wavelength (nm)",
    title=f"dispersion curve: {model_fit.dispersion(n_pix / 2):.5f} nm/px at the centre",
)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

How the other fibers were seeded: the pixel shift of each fiber's continuum-subtracted arc
spectrum relative to fiber 0 by cross-correlation (`shift_to_reference`), compared with the
true offset of the line pattern between the two traces from the simulator. A seed that is
right to within a line width is all `solve_fiber` needs; its own iterations do the rest. In the
simulation every fiber has its own small perturbation of the dispersion law, so the true offsets
are up to a few tenths of a pixel and the seeds land within about a quarter of a pixel of them,
well inside a line width; on the instrument, where the traces are tilted
and curved, the offsets are several pixels and this is the step that keeps every fiber from
having to be anchored by hand.

In [ ]:
from mastpipe.steps.wavecal import shift_to_reference

seed_shift = {s.fiber: shift_to_reference(f, fluxes[0]) for s, f in zip(arc_spectra, fluxes, strict=True)}
true_shift = {t.index: float(np.nanmedian(truth.line_pixels[t.index] - truth.line_pixels[0])) for t in traceset}
fig, ax = plt.subplots(figsize=(7, 3))
fibers = sorted(seed_shift)
ax.plot(fibers, [true_shift[f] for f in fibers], "o-", color="0.5", label="true line offset (simulator)")
ax.plot(fibers, [seed_shift[f] for f in fibers], "x", color="tab:red", ms=8, mew=2, label="cross-correlation seed")
ax.set(
    xlabel="fiber", ylabel="shift relative to fiber 0 (px)", title="seeding every fiber from the anchored one", xticks=fibers
)
ax.legend(fontsize=8)
plt.show()

### Drift from a calibration-fiber arc

Between full arcs only the calibration fibers see the lamp. For each of them the measured
lines are matched to the reference solution's lines and a linear drift in pixels (shift and
stretch) is fitted; a low-order polynomial in trace position smooths it across the detector.

In [ ]:
from mastpipe.steps.drift import fit_drift

cal_ref = next(r for r in refs if r.meta.exposure_type == "arc" and r is not arc_ref)
cal_arc = calibrate_frame(load_frame(cal_ref, effective.detector), calib)
cal_tables, _ = find_lines(cal_arc.data, cal_arc.variance, traceset, params.lines)
tables_by_fiber = {t.index: tab for t, tab in zip(traceset, cal_tables, strict=True)}
drift = fit_drift(tables_by_fiber, reference, traceset, cal_ref.meta, params.drift)

planted = truth.frames.set_index("file").loc[cal_ref.meta.source, "shift_px"]
print(f"planted shift {planted:.3f} px; model usable: {drift.usable}, scatter {drift.scatter_px:.4f} px")
drift.per_fiber

What the fit saw, for one calibration fiber: every line measured on the calibration-fiber arc
minus the position the reference solution recorded for it, against position, with the fitted
shift-and-stretch line (left). On the right, the shift of each calibration fiber against its
trace position with the smoothed model that the science fibers will read, and the shift the
simulator actually applied to this frame. The science and sky fibers are dark in a
calibration-fiber arc, so their rows of `per_fiber` hold no lines and are not usable; the fit
across the detector ignores them.

In [ ]:
fiber = int(drift.per_fiber.fiber.iloc[0])
ref_x = reference.solutions[fiber].lines.x.to_numpy()
tab = tables_by_fiber[fiber]
obs_x = tab.loc[tab.use_x, "x_center"].to_numpy()
row = drift.per_fiber.set_index("fiber").loc[fiber]
# pair each reference line with the nearest measured line once the fitted shift is undone
matched = [
    (xr, obs_x[np.argmin(np.abs(obs_x - xr - row.shift_px))])
    for xr in ref_x
    if np.min(np.abs(obs_x - xr - row.shift_px)) < params.drift.tol_px
]
xr_m, xo_m = np.array(matched).T
t = xr_m - n_pix / 2

fig, axes = plt.subplots(1, 2, figsize=(13, 3.4))
axes[0].plot(xr_m, xo_m - xr_m, ".", color="tab:red", label="matched lines")
axes[0].plot(
    xr_m,
    row.shift_px + row.stretch * t,
    "-",
    color="0.2",
    label=f"fit: shift {row.shift_px:.3f} px, stretch {row.stretch:.1e}",
)
axes[0].set(
    xlabel="reference pixel",
    ylabel="measured − reference (px)",
    title=f"fiber {fiber}: {row.n_lines} lines, rms {row.rms_px:.4f} px",
)
axes[0].legend(fontsize=8)
usable = drift.per_fiber[drift.per_fiber.usable]
dark = drift.per_fiber[~drift.per_fiber.usable]
y_grid = np.linspace(y_lo, y_hi, 50)
axes[1].plot(usable.y_center, usable.shift_px, "o", color="tab:green", label="calibration fibers (usable)")
axes[1].plot(dark.y_center, dark.shift_px, "x", color="0.6", label="dark fibers (no lines, not usable)")
axes[1].plot(y_grid, drift.shift_at(y_grid), "-", color="0.4", label="smoothed model across the detector")
axes[1].axhline(planted, color="tab:red", ls="--", label=f"planted shift {planted:.3f} px")
axes[1].set(xlabel="trace centre (row)", ylabel="shift (px)", title=f"drift model, scatter {drift.scatter_px:.4f} px")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

### Wavelengths for the science frame

A science frame is paired with calibration-fiber arcs **by time** (`pair_arcs`), and the drift
of the paired arcs is undone before the reference model gives each pixel its wavelength
(`apply_drift`). The frame's status says what happened.

In [ ]:
from mastpipe.pipeline.arcs import pair_arcs, status_for
from mastpipe.steps.drift import apply_drift

pairs_used = pair_arcs(science.meta, [(cal_ref.meta, drift)], reference, params.drift)
wavelengths, skipped = apply_drift(
    reference, pairs_used, traceset, geom, tolerance=params.trace.layout_tolerance, n_pix=n_pix
)
spectra = [replace(s, wavelength=wavelengths[s.fiber]) if s.fiber in wavelengths else s for s in spectra]
print("status:", status_for(pairs_used), "| arcs paired:", len(pairs_used), "| fibers skipped:", skipped)

What `apply_drift` produced: a wavelength for every pixel of every fiber. The upper panel is
the science and sky fiber of telescope mast01 on their pixel axes, the lower panel the same two on
their wavelength axes, where the sky emission lines of the two fibers now coincide with each
other and with the simulator's sky-line wavelengths (dotted). The bottom panel is what
`apply_drift` changed: the wavelength given to each pixel against the reference solution's own
wavelength map, in pixels, for every fiber. It is the drift at the fiber's trace position with
the sign reversed, so that a line that moved to larger columns keeps its wavelength.

In [ ]:
from mastpipe.steps.wavecal import wavelength_map

t01 = {s.role: s for s in spectra if s.telescope == "mast01"}
fig, axes = plt.subplots(3, 1, figsize=(11, 8))
for role, color in (("science", "tab:red"), ("sky", "tab:blue")):
    s = t01[role]
    axes[0].plot(s.pixel, s.flux, lw=0.7, color=color, label=f"fiber {s.fiber} ({role})")
    axes[1].plot(s.wavelength, s.flux, lw=0.7, color=color, label=f"fiber {s.fiber} ({role})")
for w in truth.sky_lines_nm:
    axes[1].axvline(w, color="0.3", ls=":", lw=0.8)
axes[0].set(xlabel="pixel", ylabel="ADU", title="telescope mast01 before: the two fibers on their own pixel axes")
axes[1].set(
    xlabel="wavelength (nm)",
    ylabel="ADU",
    title="after: on wavelength axes, the sky lines align (dotted: the simulator's sky lines)",
)
for fiber, lam in sorted(wavelengths.items()):
    ref_model = model_from_solution(reference.solutions[fiber], geom)
    axes[2].plot(
        x,
        (lam - wavelength_map(reference.solutions[fiber], geom)) / ref_model.dispersion(x),
        lw=0.8,
        color=role_color.get(next(t.role for t in traceset if t.index == fiber), "0.5"),
        label=f"fiber {fiber}",
    )
axes[2].axhline(-planted, color="tab:red", ls="--", lw=0.8, label=f"−planted shift ({-planted:.3f} px)")
axes[2].set(
    xlabel="pixel",
    ylabel="(applied − reference) / dispersion (px)",
    title="the drift correction apply_drift folded into every fiber's wavelength map",
)
axes[2].legend(fontsize=7, ncol=5)
for ax in axes[:2]:
    ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

## Stage 4 — sky subtraction

Each science fiber minus the sky fiber of the same telescope (from the layout), scaled by the
flat's relative throughput and resampled in wavelength space. The raw extraction stays; the
result is a new version of the spectrum.

One thing to know when reading the figure: the simulator puts **no** sky light into the science
fibers, so here the subtraction carves negative sky lines out of the science spectrum. On the
sky the science fiber carries the same sky as its neighbour and the lines cancel; what to look
at is that the sky is scaled, resampled onto the science fiber's wavelengths and stored next to
the raw extraction, never in place of it.

In [ ]:
from mastpipe.steps.sky import subtract_sky_all

spectra, reasons = subtract_sky_all(spectra, traceset, throughput, params.sky)
print("not subtracted:", reasons)
s = next(s for s in spectra if s.telescope == "mast01" and s.role == "science")
fig, axes = plt.subplots(1, 2, figsize=(13, 3.4), gridspec_kw={"width_ratios": [3, 1]})
for ax in axes:
    ax.plot(s.wavelength, s.flux, lw=0.8, color="0.5", label="raw extraction")
    ax.plot(s.wavelength, s.sky, lw=0.8, color="tab:blue", label="sky (resampled, scaled)")
    ax.plot(s.wavelength, s.versions["skysub"].flux, lw=0.8, color="tab:red", label="sky-subtracted")
    ax.set(xlabel="wavelength (nm)", ylabel="ADU")
axes[0].set(title=f"fiber {s.fiber}, telescope mast01, target {s.target}")
axes[0].legend(fontsize=8)
w_sky = truth.sky_lines_nm[1]
axes[1].set(xlim=(w_sky - 0.4, w_sky + 0.4), title=f"zoom on the sky line at {w_sky:.1f} nm")
plt.tight_layout()
plt.show()

## Combination

The science fibers of one target in the exposure are combined on a common wavelength grid
with inverse-variance weights, every version separately; excluded contributors are listed
with a reason.

In [ ]:
from mastpipe.steps.combine import combine_spectra, group_by_target

groups = group_by_target(spectra)
target, members = next(iter(groups.items()))
combined = combine_spectra(members, target, params.combine)
print(target, "from", len(members), "fibers;", "versions:", list(combined.versions))
fig, axes = plt.subplots(1, 2, figsize=(13, 3.4), gridspec_kw={"width_ratios": [3, 1]})
for m in members:
    axes[0].plot(m.wavelength, m.versions["skysub"].flux, lw=0.5, alpha=0.5, label=f"fiber {m.fiber} ({m.telescope})")
axes[0].plot(combined.wavelength, combined.versions["skysub"].flux, lw=0.9, color="k", label="combined, sky-subtracted")
axes[0].set(xlabel="wavelength (nm)", ylabel="ADU", title=f"combined spectrum of {target} and its contributors")
axes[0].legend(fontsize=8)
c = combined.contributors
axes[1].bar(c.fiber.astype(str) + "\n" + c.telescope, c.weight, color="tab:gray")
axes[1].set(ylabel="weight", title="inverse-variance weights")
plt.tight_layout()
plt.show()
combined.contributors

## Writing and reading a product

A product is a FITS file with one row per fiber, the raw extraction and every version as
columns, the drift table, and provenance cards. The reader gives back the same dataclasses.

In [ ]:
from mastpipe import __version__
from mastpipe.cli import main
from mastpipe.config.loader import config_hash
from mastpipe.core.model import ExposureSpectra
from mastpipe.io.products import product_name, read_spectra, write_spectra
from mastpipe.pipeline.provenance import Provenance

es = ExposureSpectra(
    meta=science.meta,
    spectra=spectra,
    wavelength_status=status_for(pairs_used),
    reference_id=reference.identifier,
    reference_hash=reference.hash,
    drift_arcs=tuple(sorted(m.checksum for m, _ in [(cal_ref.meta, drift)])),
)
prov = Provenance(
    pipeline_version=__version__,
    config_hash=config_hash(effective),
    inputs=[science.meta.checksum],
    calibrations=["bias", "dark", "background", "flat"],
    wavelength_status=es.wavelength_status,
    reference_id=es.reference_id,
    reference_hash=es.reference_hash,
    drift_arcs=list(es.drift_arcs),
)
path = work / product_name(science.meta, "spec")
write_spectra(path, es, prov, drift=drift, weights={cal_ref.meta.checksum: 1.0})

back, back_prov, back_drift = read_spectra(path)
print(path.name, "|", len(back.spectra), "fibers |", back.wavelength_status, "|", sorted(back.spectra[1].versions))
main(["show", str(path)])

The file itself: its extensions and the columns of the `SPECTRA` table, then one row's
`FLUX_SKYSUB` column read with plain astropy against the in-memory spectrum it came from, to
show that nothing is lost on the way to disk.

In [ ]:
with fits.open(path) as hdul:
    hdul.info()
    cols = hdul["SPECTRA"].columns.names
    row = hdul["SPECTRA"].data[[int(f) for f in hdul["SPECTRA"].data["FIBER"]].index(s.fiber)]
    disk_flux, disk_wave, disk_mask = row["FLUX_SKYSUB"].copy(), row["WAVELENGTH"].copy(), row["MASK_SKYSUB"].copy()
print("SPECTRA columns:", ", ".join(cols))

fig, axes = plt.subplots(2, 1, figsize=(11, 4.5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
axes[0].plot(s.wavelength, s.versions["skysub"].flux, lw=1.6, color="0.7", label="in memory")
axes[0].plot(
    disk_wave,
    np.where(disk_mask, np.nan, disk_flux),
    lw=0.7,
    color="tab:red",
    label="read back from the FITS table (masked pixels blanked)",
)
axes[0].set(ylabel="ADU", title=f"{path.name}, row of fiber {s.fiber}: FLUX_SKYSUB against WAVELENGTH")
axes[0].legend(fontsize=8)
axes[1].plot(disk_wave, disk_flux - s.versions["skysub"].flux, lw=0.7, color="0.3")
axes[1].set(xlabel="wavelength (nm)", ylabel="disk − memory")
plt.tight_layout()
plt.show()

## Where to go next

- `mastpipe quicklook` runs exactly this chain for every frame of a folder, with the
  calibration library, idempotence and QC files; [Inspecting a night's products](inspecting-products.ipynb)
  reads what it writes.
- The functions used here are documented in the [API reference](../api/index.md); their
  methods and the choices behind them in [How it works](../how-it-works/index.md).
- To run on a real night, replace the simulated folder with yours and the hand-made layout
  with the instrument's preset (`load_layout("highspec", "survey45")`).